# 4.2 · 为大模型设计好用的工具

**Designing tools an LLM can use well**

> **学完本课你能：** 设计、检查并重构一套 MCP 工具——名称、描述、模式、注解、可操作的错误、分页和状态句柄——让模型选对工具、正确调用，并能从错误中恢复。

官方阅读材料：[Spec: Tools (stateful tools, errors)](https://modelcontextprotocol.io/specification/2026-07-28/server/tools) · [Build with Agent Skills](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-with-agent-skills) · [Client best practices](https://modelcontextprotocol.io/docs/2026-07-28/develop/clients/client-best-practices)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个工单系统数据集和两套工具集
两个服务器在同一份数据上暴露<em>相同</em>的能力（Capability）。其中一个的设计方式，就像人们匆忙包装接口时常做的那样。另一个则遵循本课的原则。请把两者都放在眼前：之后的所有内容都是在比较它们。

In [ ]:
import json
import logging
import random
import warnings
from datetime import date, timedelta

warnings.filterwarnings("ignore", category=DeprecationWarning)
# The SDK logs every failed tool call server-side (with a traceback for crashes). That is the *server*
# log, which the model never sees; we silence it so the notebook shows what the client receives.
logging.getLogger("mcp").setLevel(logging.CRITICAL)
rng = random.Random(7)
CUSTOMERS = ["Acme", "Globex", "Initech", "Umbrella", "Hooli"]
TICKETS = {}
for i in range(1, 1201):
    tid = f"T-{i:04d}"
    TICKETS[tid] = {
        "id": tid,
        "title": rng.choice(["Login fails", "Invoice wrong", "Slow dashboard", "Export broken", "API 500s"]),
        "status": rng.choices(["open", "pending", "closed"], [5, 2, 3])[0],
        "priority": rng.choices(["low", "normal", "high", "urgent"], [3, 5, 2, 1])[0],
        "customer": rng.choice(CUSTOMERS),
        "created": (date(2026, 1, 1) + timedelta(days=rng.randrange(270))).isoformat(),
        "body": " ".join(rng.choice(["The", "user", "reports", "that", "the", "page", "times", "out", "again",
                                     "after", "upgrading", "to", "v2", "and", "clearing", "cache"]) for _ in range(120)),
    }
print(len(TICKETS), "tickets; example:", {k: v for k, v in TICKETS["T-0001"].items() if k != "body"})

In [ ]:
from mcp.server import MCPServer

bad = MCPServer("helpdesk-bad")


@bad.tool()
def query(q: str, opts: dict | None = None) -> str:
    """Query."""
    opts = opts or {}
    rows = [t for t in TICKETS.values() if q.lower() in t["title"].lower()]
    if "status" in opts:
        if opts["status"] not in ("open", "pending", "closed"):
            raise ValueError(f"bad status {opts['status']}")
        rows = [t for t in rows if t["status"] == opts["status"]]
    if "since" in opts:
        since = date.fromisoformat(opts["since"])          # raises ValueError on 03/01/2026
        rows = [t for t in rows if date.fromisoformat(t["created"]) >= since]
    return json.dumps(rows)                                    # everything, bodies included


@bad.tool()
def do(action: str, id: str) -> str:
    if action not in ("close", "reopen"):
        raise ValueError("bad action")
    if TICKETS[id]["status"] == "closed" and action == "close":
        raise ValueError("already closed")                     # retries blow up
    TICKETS[id]["status"] = "closed" if action == "close" else "open"
    return "ok"


@bad.tool()
def getTicketData2(x: str) -> dict:
    """Gets data."""
    return TICKETS[x]


print("defined", bad.name, "with tools: query, do, getTicketData2")

In [ ]:
import base64
import hashlib
from typing import Annotated, Literal

from pydantic import BaseModel, Field

from mcp.server.mcpserver.exceptions import ToolError
from mcp_types import ToolAnnotations

Status = Literal["open", "pending", "closed"]
Priority = Literal["low", "normal", "high", "urgent"]


class TicketSummary(BaseModel):
    id: str
    title: str
    status: Status
    priority: Priority
    customer: str
    created: str


class SearchPage(BaseModel):
    items: list[TicketSummary]
    total: int = Field(description="Number of tickets matching the filters")
    next_cursor: str | None = Field(description="Pass back as `cursor` for the next page; null when done")


class TicketDetail(TicketSummary):
    body: str | None = Field(description="Ticket text, truncated to 400 characters unless include_body=True")
    body_truncated: bool


class CloseResult(BaseModel):
    id: str
    status: Status
    changed: bool = Field(description="False if the ticket was already closed (safe to retry)")


def parse_iso_date(value: str, field: str) -> date:
    try:
        return date.fromisoformat(value)
    except ValueError:
        raise ToolError(f"Invalid {field} {value!r}: use an ISO date YYYY-MM-DD, e.g. 2026-03-01.") from None


def make_cursor(offset: int, filters: str) -> str:
    return base64.urlsafe_b64encode(json.dumps({"o": offset, "f": filters}).encode()).decode()


def read_cursor(cursor: str, filters: str) -> int:
    try:
        data = json.loads(base64.urlsafe_b64decode(cursor.encode()))
    except Exception:
        raise ToolError("Invalid cursor: pass back next_cursor exactly as returned, or omit it to start over.") from None
    if data["f"] != filters:
        raise ToolError("This cursor belongs to a search with different filters. Repeat the search without a cursor.")
    return data["o"]


good = MCPServer("helpdesk")
READ_ONLY = ToolAnnotations(readOnlyHint=True, openWorldHint=False)


@good.tool(title="Search tickets", annotations=READ_ONLY)
def tickets_search(
    status: Annotated[Status, Field(description="Ticket status to match")] = "open",
    priority: Annotated[Priority | None, Field(description="Only this priority; omit for any")] = None,
    customer: Annotated[str | None, Field(description="Exact customer name, e.g. Acme")] = None,
    created_after: Annotated[str | None, Field(description="ISO date YYYY-MM-DD; only tickets created on/after it")] = None,
    limit: Annotated[int, Field(ge=1, le=50, description="Page size (1-50)")] = 10,
    cursor: Annotated[str | None, Field(description="next_cursor from a previous page of the same search")] = None,
) -> SearchPage:
    """Search helpdesk tickets by status, priority, customer and creation date.

    Returns at most `limit` short summaries (no ticket text) plus `total` and `next_cursor`.
    Use tickets_get for the full text of one ticket, and tickets_close to close it.
    """
    since = parse_iso_date(created_after, "created_after") if created_after else None
    filters = json.dumps([status, priority, customer, created_after])
    offset = read_cursor(cursor, filters) if cursor else 0
    rows = [t for t in TICKETS.values() if t["status"] == status
            and (priority is None or t["priority"] == priority)
            and (customer is None or t["customer"] == customer)
            and (since is None or date.fromisoformat(t["created"]) >= since)]
    page = rows[offset:offset + limit]
    more = offset + limit < len(rows)
    return SearchPage(items=[TicketSummary(**{k: t[k] for k in TicketSummary.model_fields}) for t in page],
                      total=len(rows), next_cursor=make_cursor(offset + limit, filters) if more else None)


@good.tool(title="Get ticket", annotations=READ_ONLY)
def tickets_get(
    ticket_id: Annotated[str, Field(pattern=r"^T-\d{4}$", description="Ticket id like T-0042")],
    include_body: Annotated[bool, Field(description="Return the full text instead of the first 400 chars")] = False,
) -> TicketDetail:
    """Get one ticket with its text. Find ids with tickets_search."""
    t = TICKETS.get(ticket_id)
    if t is None:
        raise ToolError(f"Ticket {ticket_id} does not exist. Use tickets_search to find valid ids.")
    body = t["body"] if include_body else t["body"][:400]
    return TicketDetail(**{k: t[k] for k in TicketSummary.model_fields}, body=body,
                        body_truncated=len(body) < len(t["body"]))


@good.tool(title="Close ticket",
           annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=False, idempotentHint=True,
                                       openWorldHint=False))
def tickets_close(
    ticket_id: Annotated[str, Field(pattern=r"^T-\d{4}$", description="Ticket id like T-0042")],
    resolution: Annotated[Literal["fixed", "wont_fix", "duplicate"], Field(description="Why it is closed")],
) -> CloseResult:
    """Close a ticket with a resolution. Idempotent: closing a closed ticket succeeds with changed=false."""
    t = TICKETS.get(ticket_id)
    if t is None:
        raise ToolError(f"Ticket {ticket_id} does not exist. Use tickets_search to find valid ids.")
    changed = t["status"] != "closed"
    t["status"], t["resolution"] = "closed", resolution
    return CloseResult(id=ticket_id, status="closed", changed=changed)


print("defined", good.name, "with tools: tickets_search, tickets_get, tickets_close")

### 2. 一个工具定义检查器
一个启发式的检查器（linter）让这些指导原则变得可度量。它只读取客户端能看到的内容，即 `tools/list` 的结果。每条规则都标注了来源：规范规则（`[spec]`）和通用实践（`[practice]`）。

In [ ]:
import re

from mcp import Client

NAME_RE = re.compile(r"^[A-Za-z0-9_.\-]{1,128}$")
CLOSED_SET_HINTS = ("status", "priority", "action", "mode", "type", "kind", "resolution", "level")


def lint_tool(tool, prefixes: set[str]) -> list[tuple[int, str]]:
    """Return (penalty, message) pairs for one tool definition."""
    findings = []
    if not NAME_RE.match(tool.name):
        findings.append((15, "[spec] name uses characters outside A-Za-z0-9_-. or is too long"))
    if len(prefixes) > 1 or "_" not in tool.name:
        findings.append((5, "[practice] no shared service prefix (namespace your tools)"))
    if tool.name.split("_")[-1] in {"do", "run", "query", "process", "data", "data2"} or tool.name in {"do", "query"}:
        findings.append((10, "[practice] generic name says nothing about intent"))
    desc = (tool.description or "").strip()
    if len(desc) < 40:
        findings.append((15, f"[practice] description too short ({len(desc)} chars) to tell the model when to use it"))
    props = tool.input_schema.get("properties", {})
    for pname, schema in props.items():
        if "description" not in schema:
            findings.append((4, f"[practice] parameter '{pname}' has no description"))
        typed = any(k in schema for k in ("type", "enum", "anyOf", "$ref", "const"))
        if not typed or (schema.get("type") == "object" and "properties" not in schema) or any(
                s.get("type") == "object" and "properties" not in s for s in schema.get("anyOf", [])):
            findings.append((8, f"[practice] parameter '{pname}' is an untyped/free-form object"))
        is_string = schema.get("type") == "string" or any(s.get("type") == "string" for s in schema.get("anyOf", []))
        has_enum = "enum" in schema or any("enum" in s for s in schema.get("anyOf", []))
        if is_string and not has_enum and any(h in pname.lower() for h in CLOSED_SET_HINTS):
            findings.append((6, f"[practice] '{pname}' looks like a closed set but has no enum"))
    if not tool.output_schema:
        findings.append((10, "[spec/docs] no outputSchema (code-mode hosts can't type the result)"))
    if not tool.annotations or tool.annotations.read_only_hint is None:
        findings.append((8, "[spec] annotations missing: defaults say destructive + open-world"))
    return findings


async def lint_server(server) -> int:
    async with Client(server) as client:
        tools = (await client.list_tools()).tools
        wire = json.dumps([t.model_dump(by_alias=True, exclude_none=True, mode="json") for t in tools])
    prefixes = {t.name.split("_")[0] for t in tools}
    total = 0
    print(f"== {server.name}: {len(tools)} tools, ~{len(wire) // 4} tokens of definitions")
    for t in tools:
        findings = lint_tool(t, prefixes)
        score = max(0, 100 - sum(p for p, _ in findings))
        total += score
        print(f"  {t.name:16} score {score:3}")
        for _, msg in findings:
            print(f"      - {msg}")
    return total // len(tools)


print("average bad :", await lint_server(bad))
print()
print("average good:", await lint_server(good))

良好的工具集在定义上花费的 token <em>更多</em>。这是正确的取舍：几百个 token 的精确模式（Schema）能省下整个白白浪费的往返。你必须避免的，是把 token 花在<em>结果</em>上（第 4 节）。

### 3. 让智能体能够恢复的错误
真实的模型会阅读错误文本并再次尝试。这个脚本化的智能体（Agent）代替了模型：它只能修正消息中解释清楚了的问题。它会三种修复：被告知格式时重新格式化日期，列出有效值时选择最接近的有效值，否则就放弃。

In [ ]:
import difflib


def repair(args: dict, error: str) -> dict | None:
    """Return corrected arguments, or None if the error gives nothing to act on."""
    fixed = json.loads(json.dumps(args))
    flat = fixed.get("opts", fixed)
    if "YYYY-MM-DD" in error:
        for key, value in flat.items():
            if isinstance(value, str) and re.fullmatch(r"\d\d/\d\d/\d{4}", value):
                m, d, y = value.split("/")
                flat[key] = f"{y}-{m}-{d}"
                return fixed
    choices = re.findall(r"'([a-z_]+)'", error.split("Input should be", 1)[-1]) if "Input should be" in error else []
    if choices:
        for key, value in flat.items():
            if isinstance(value, str) and value not in choices:
                close = difflib.get_close_matches(value.lower(), choices, n=1)
                if close:
                    flat[key] = close[0]
                    return fixed
    return None


async def agent(server, tool: str, args: dict, max_tries: int = 4):
    async with Client(server) as client:
        for attempt in range(1, max_tries + 1):
            r = await client.call_tool(tool, args)
            if not r.is_error:
                summary = r.structured_content or {}
                total = summary.get("total", f"{len(json.loads(r.content[0].text))} rows")
                print(f"  try {attempt}: OK   {json.dumps(args)} -> total={total}")
                return
            message = r.content[0].text.replace("\n", " ")[:150]
            print(f"  try {attempt}: ERR  {json.dumps(args)}\n           model reads: {message!r}")
            args = repair(args, r.content[0].text)
            if args is None:
                print("           nothing actionable in that message -> agent gives up")
                return


print("bad server (ValueError -> masked by the SDK):")
await agent(bad, "query", {"q": "", "opts": {"status": "Open", "since": "03/01/2026"}})
print("\ngood server (enum + ToolError with format and example):")
await agent(good, "tickets_search", {"status": "Open", "created_after": "03/01/2026"})

在糟糕的服务器上发生了两件事。模型从未得知<em>哪里</em>错了，因为 SDK 隐藏了普通异常。而且即使是原始消息（“bad status Open”）也没有列出有效值。在良好的服务器上，enum 在函数运行前就被校验，日期错误中带有格式和示例。

### 4. 大结果：token 预算、分页、截断
糟糕的 `query` 会返回每一个匹配的工单及其正文。良好的搜索则返回一页摘要，外加 `total` 和一个不透明游标（Opaque Cursor）。按大约 4 个字符折合 1 个 token 的粗略估算比较它们的大小，然后用游标遍历所有页面。

In [ ]:
def tokens(text: str) -> int:
    return len(text) // 4


async with Client(bad) as client:
    r = await client.call_tool("query", {"q": "", "opts": {"status": "open"}})
    print(f"bad  query(status=open): {tokens(r.content[0].text):>7,} tokens in ONE result")

async with Client(good) as client:
    r = await client.call_tool("tickets_search", {"status": "open"})
    page = r.structured_content
    print(f"good tickets_search    : {tokens(r.content[0].text):>7,} tokens, {len(page['items'])} of {page['total']} items")
    print("     next_cursor (opaque):", page["next_cursor"][:40], "...")

    seen, cursor, calls = 0, None, 0
    while True:
        args = {"status": "open", "priority": "urgent", "limit": 50} | ({"cursor": cursor} if cursor else {})
        r = await client.call_tool("tickets_search", args)
        calls += 1
        seen += len(r.structured_content["items"])
        cursor = r.structured_content["next_cursor"]
        if not cursor:
            break
    print(f"\nwalked {seen} urgent open tickets in {calls} pages")

    wrong = await client.call_tool("tickets_search", {"status": "pending", "cursor": page["next_cursor"]})
    print("reusing a cursor with other filters ->", wrong.content[0].text)

    detail = await client.call_tool("tickets_get", {"ticket_id": "T-0001"})
    d = detail.structured_content
    print(f"\ntickets_get default: body {len(d['body'])} chars, body_truncated={d['body_truncated']} "
          "(the field tells the model how to get the rest)")

### 5. 幂等的写操作经得起重试
智能体和宿主会在超时后重试。在两个服务器上各把同一个工单关闭两次。

In [ ]:
first_open = next(t["id"] for t in TICKETS.values() if t["status"] == "open")
second_open = next(t["id"] for t in TICKETS.values() if t["status"] == "open" and t["id"] != first_open)

async with Client(bad) as client:
    for attempt in (1, 2):
        r = await client.call_tool("do", {"action": "close", "id": first_open})
        print(f"bad  do(close) attempt {attempt}: isError={r.is_error} {r.content[0].text!r}")

async with Client(good) as client:
    for attempt in (1, 2):
        r = await client.call_tool("tickets_close", {"ticket_id": second_open, "resolution": "fixed"})
        print(f"good tickets_close attempt {attempt}: isError={r.is_error} {r.structured_content}")
    t = next(t for t in (await client.list_tools()).tools if t.name == "tickets_close")
    print("\nannotations on the wire:", t.annotations.model_dump(by_alias=True, exclude_none=True))

### 6. 确定性的顺序让缓存保持命中
SDK 按注册顺序列出工具。只有<em>你的</em>注册顺序是确定的，它才是确定的。这里有一个服务器通过遍历一个 `set` 来注册工具，这是一个经典错误。Python 在每个进程中对字符串哈希进行随机化，所以两个“副本”（两个 `PYTHONHASHSEED` 不同的进程）会公布不同的顺序。不同的 `tools` 数组意味着不同的提示前缀，提供商的提示缓存（Prompt Caching）就会未命中。

In [ ]:
import subprocess
import sys
import textwrap

SNIPPET = textwrap.dedent('''
    import anyio, hashlib, json, sys
    from mcp import Client
    from mcp.server import MCPServer
    names = {"tickets_search", "tickets_get", "tickets_close", "tickets_assign", "tickets_comment"}
    srv = MCPServer("helpdesk")
    for name in (sorted(names) if sys.argv[1] == "sorted" else names):
        srv.add_tool(lambda: "ok", name=name, description=f"{name} tool")
    async def main():
        async with Client(srv) as c:
            tools = [t.model_dump(by_alias=True, exclude_none=True, mode="json") for t in (await c.list_tools()).tools]
        print(hashlib.sha256(json.dumps(tools).encode()).hexdigest()[:12], [t["name"] for t in tools])
    anyio.run(main)
''')

for mode in ("set", "sorted"):
    print(f"registration from a {mode}:")
    for seed in ("1", "2"):
        out = subprocess.run([sys.executable, "-c", SNIPPET, mode], capture_output=True, text=True,
                             env={"PYTHONHASHSEED": seed, "PATH": ""}).stdout.strip()
        print(f"  replica with PYTHONHASHSEED={seed}: {out}")

### 7. 没有会话的状态：显式句柄
一个分诊工作流跨越多次调用。`triage_start` 不把队列藏在连接里，而是返回一个不透明的高熵句柄（状态句柄（State Handle）），其生命周期<strong>写在它的描述中</strong>。之后的每次调用都把它传回来。我们使用两个<em>独立的</em>客户端连接，以表明没有任何东西依赖于连接。然后我们让句柄过期，观察错误信息如何告诉智能体进行恢复。

In [ ]:
import time
import uuid

TRIAGE: dict[str, dict] = {}          # in production: a shared store (Redis, DB) so any replica can serve
TTL_SECONDS = 15 * 60
clock = {"skew": 0.0}                 # lets the notebook fast-forward time


def now() -> float:
    return time.time() + clock["skew"]


@good.tool(title="Start triage", annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=False,
                                                             idempotentHint=False, openWorldHint=False))
def triage_start(priority: Annotated[Priority, Field(description="Queue to triage")] = "urgent") -> dict[str, str | int]:
    """Start a triage session over open tickets of one priority.

    Returns a triage_id to pass to triage_next. Sessions expire after 15 minutes of inactivity;
    if a triage_id is rejected as expired, call triage_start again.
    """
    queue = [t["id"] for t in TICKETS.values() if t["status"] == "open" and t["priority"] == priority]
    triage_id = f"trg_{uuid.uuid4().hex}"
    TRIAGE[triage_id] = {"queue": queue, "pos": 0, "touched": now()}
    return {"triage_id": triage_id, "queued": len(queue)}


@good.tool(title="Next ticket in triage", annotations=READ_ONLY)
def triage_next(triage_id: Annotated[str, Field(description="Handle returned by triage_start")]) -> dict[str, str | int]:
    """Return the next ticket id in a triage session and how many remain."""
    state = TRIAGE.get(triage_id)
    if state is None or now() - state["touched"] > TTL_SECONDS:
        TRIAGE.pop(triage_id, None)
        raise ToolError(f"Triage session {triage_id[:12]}... is unknown or expired (sessions last 15 minutes). "
                        "Call triage_start to begin a new one.")
    state["touched"] = now()
    if state["pos"] >= len(state["queue"]):
        return {"ticket_id": "", "remaining": 0}
    tid = state["queue"][state["pos"]]
    state["pos"] += 1
    return {"ticket_id": tid, "remaining": len(state["queue"]) - state["pos"]}


async with Client(good) as conn_a:
    started = (await conn_a.call_tool("triage_start", {"priority": "urgent"})).structured_content
    print("connection A: triage_start ->", started)
handle = started["triage_id"]

async with Client(good) as conn_b:                     # a different connection (or replica)
    for _ in range(2):
        print("connection B: triage_next ->", (await conn_b.call_tool("triage_next", {"triage_id": handle})).structured_content)

    clock["skew"] += 16 * 60                           # 16 minutes pass
    expired = await conn_b.call_tool("triage_next", {"triage_id": handle})
    print("\nafter 16 min ->", expired.is_error, expired.content[0].text)
    if "triage_start" in expired.content[0].text:      # the agent follows the instruction in the error
        fresh = (await conn_b.call_tool("triage_start", {"priority": "urgent"})).structured_content
        print("agent recovers: new handle", fresh["triage_id"][:12] + "...", "| queued", fresh["queued"])

### 8. 可选：让 Claude 来驱动良好的工具集
设置 `ANTHROPIC_API_KEY` 后，Claude 会拿到良好的工具集（名称、描述、模式与公布的完全一致），以及一个需要过滤、分页和一次写操作的任务。观察它选择了哪些工具，以及它是否能在几次调用之内完成。

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with Client(good) as client:
        tool_defs = [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                     for t in (await client.list_tools()).tools]
        messages = [{"role": "user", "content": "Close every open urgent ticket from Hooli created after "
                                                "2026-09-01 as duplicate, then tell me how many you closed."}]
        for _ in range(12):
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tool_defs, messages=messages)
            if response.stop_reason != "tool_use":
                break
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    r = await client.call_tool(block.name, block.input)
                    print(f"Claude -> {block.name}({json.dumps(block.input)[:90]}) isError={r.is_error}")
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": r.content[0].text, "is_error": bool(r.is_error)})
            messages.append({"role": "user", "content": results})
        print("Claude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude use tickets_search -> tickets_close.")

### 动手练习
1. 一步步修复糟糕的服务器：重命名工具并加上命名空间，添加描述，用带类型的参数和 enum 替换 `opts: dict`，并添加注解。每次修改后重新运行第 2 节，观察分数的变化。
2. 把 `parse_iso_date` 改为抛出带相同消息的普通 `ValueError`，然后重新运行第 3 节。模型现在看到了什么？为什么？
3. 添加一个 `tickets_create(title, customer, priority, idempotency_key)` 工具，当键重复时返回已有的工单。用相同的键调用它两次来证明这一点。它应该带有哪些注解？
4. 为 `triage_next` 添加授权检查：在 `triage_start` 中保存一个 `owner`（假装它来自请求的凭据），并用一条可操作的错误拒绝其他所有者。